In [ ]:
"""
DINOv3 Canopy Height Model Inference Script
Optimized for JupyterHub multi-GPU inference on large geospatial imagery
"""

# Essential imports
import os
import warnings
import glob
import numpy as np
import torch
import torch.nn as nn
from PIL import Image
import torchvision.transforms as transforms
from transformers import AutoModel
import rasterio
from tiler import Tiler, Merger
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm
from tqdm import tqdm
import time
from osgeo import gdal
import traceback

# ==========================================
# 1. CONFIGURATION (Edit these as needed)
# ==========================================

# EDIT: point this at your own directory of 8-band Planet SuperDove SR scenes
# (same band convention as documented in README.md)
INPUT_DIR = "./data/planet_scenes/"
OUTPUT_DIR = "./output/"
# EDIT: place the checkpoint file here -- it's too large to be tracked in this
# repo, so it must be transferred separately (see README.md)
CHECKPOINT_PATH = "./checkpoint/best_dinov3_fulldataset_customloss.pth"
# Generated by running np_stats.ipynb on your own imagery first
PLANET_STATS_FILE = "./numpy_stats/planet_stats_combined_planet_8b_sr.npy"
PLANET_SCALED_MEANS_FILE = "./numpy_stats/channel_means_planet_nrg_scaled01.npy"
PLANET_SCALED_STDS_FILE = "./numpy_stats/channel_stds_planet_nrg_scaled01.npy"

BATCH_SIZE = 1024
IMG_SIZE = 64
OVERLAP = 0.5
VISUALIZE_FIRST = True

# Create output directory if it doesn't exist
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ==========================================
# 2. SYSTEM OPTIMIZATIONS
# ==========================================

print("🔧 Setting CUDA optimizations...")
torch.backends.cudnn.benchmark = True
torch.backends.cudnn.deterministic = False
torch.backends.cudnn.enabled = True

os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'max_split_size_mb:128'
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'
warnings.filterwarnings("ignore")

def monitor_gpu_usage():
    if torch.cuda.is_available():
        print("🔍 GPU Memory Status:")
        for i in range(torch.cuda.device_count()):
            allocated = torch.cuda.memory_allocated(i) / 1e9
            reserved = torch.cuda.memory_reserved(i) / 1e9
            total = torch.cuda.get_device_properties(i).total_memory / 1e9
            utilization = (allocated / total) * 100
            print(f"  GPU {i}: {allocated:.2f}GB/{total:.1f}GB ({utilization:.1f}%) allocated, {reserved:.2f}GB reserved")
    else:
        print("❌ No CUDA GPUs available for monitoring")

def get_channels(input_bands):
    if input_bands == 'nrg':
        return [3, 2, 1] 
    elif input_bands == 'rgb':
        return [2, 1, 0]
    else:
        return [0, 1, 2, 3]

def create_custom_binned_colormap(colors, vmax=35):
    boundaries = [0, 0.001, .5, 1, 2, 3, 5, 10, vmax]
    forest_ht_cmap = ListedColormap(colors)
    norm = BoundaryNorm(boundaries, len(colors))
    return forest_ht_cmap, norm, boundaries

# ==========================================
# 3. MODEL ARCHITECTURE
# ==========================================

class ChannelAttention(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        self.attention = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(channels, channels//reduction, 1),
            nn.ReLU(),
            nn.Conv2d(channels//reduction, channels, 1),
            nn.Sigmoid()
        )
    
    def forward(self, x):
        return x * self.attention(x)

class DINOv3DepthHead(nn.Module):
    def __init__(self, model_name="facebook/dinov3-vitl16-pretrain-sat493m", 
             freeze_backbone=True, output_channels=1, token=None, input_bands='rgb',
             training_config=None):
        
        super().__init__()
        
        self.training_config = training_config or {}
        self.dropout_rate = self.training_config.get('dropout_rate', 0.1)
        self.output_channels = output_channels
        
        if token:
            self.backbone = AutoModel.from_pretrained(model_name, token=token)
        else:
            self.backbone = AutoModel.from_pretrained(model_name)
    
        if input_bands == 'nrg':
            self._modify_input_weights_for_nrg()
        
        self._determine_model_dimensions()
        
        if freeze_backbone:
            for param in self.backbone.parameters():
                param.requires_grad = False
            self._freeze_backbone = True
        
        print(f"DINOv3 DepthHead initialized:")
        print(f"  Model: {model_name}")
        print(f"  Input bands: {input_bands}")
        print(f"  Embedding dim: {self.embed_dim}")
        print(f"  Total tokens: {self.total_tokens}")
        
        self.depth_head = self._build_improved_decoder()
        self._initialize_output_layer(target_percentile=8.42)
    
    def _determine_model_dimensions(self):
        dummy_input = torch.randn(1, 3, 64, 64)
        with torch.no_grad():
            outputs = self.backbone(dummy_input)
            features = outputs.last_hidden_state
        
        self.total_tokens = features.shape[1]
        self.embed_dim = features.shape[2]
        self.patches_per_side = 64 // 16 
        self.num_spatial_patches = self.patches_per_side ** 2
        
        non_spatial_tokens = self.total_tokens - self.num_spatial_patches
        if non_spatial_tokens == 1:
            self.spatial_start = 1
            self.spatial_end = self.total_tokens
        elif non_spatial_tokens == 5:
            self.spatial_start = 5
            self.spatial_end = self.total_tokens
        else:
            self.spatial_start = self.total_tokens - self.num_spatial_patches
            self.spatial_end = self.total_tokens

    def _modify_input_weights_for_nrg(self):
        print("Modifying input weights for NIR-Red-Green bands...")
        for name, module in self.backbone.named_modules():
            if isinstance(module, nn.Conv2d) and module.in_channels == 3:
                with torch.no_grad():
                    original_weights = module.weight.data.clone()
                    new_weights = torch.zeros_like(original_weights)
                    new_weights[:, 0, :, :] = original_weights[:, 2, :, :] 
                    new_weights[:, 1, :, :] = original_weights[:, 2, :, :] 
                    new_weights[:, 2, :, :] = original_weights[:, 1, :, :]
                    module.weight.data = new_weights
                break

    def _build_improved_decoder(self):
        class ResidualBlock(nn.Module):
            def __init__(self, channels, use_attention=True, dropout_rate=0.1):
                super().__init__()
                self.conv1 = nn.Conv2d(channels, channels, kernel_size=3, padding=1)
                self.bn1 = nn.BatchNorm2d(channels)
                self.conv2 = nn.Conv2d(channels, channels, kernel_size=3, padding=1)
                self.bn2 = nn.BatchNorm2d(channels)
                self.activation = nn.LeakyReLU(0.1, inplace=True)
                self.dropout = nn.Dropout2d(dropout_rate)
                self.attention = ChannelAttention(channels) if use_attention else nn.Identity()
                
            def forward(self, x):
                residual = x
                out = self.activation(self.bn1(self.conv1(x)))
                out = self.dropout(out)
                out = self.bn2(self.conv2(out))
                out = self.attention(out)
                out += residual
                return self.activation(out)
               
        dropout_rate = self.dropout_rate
        layers = nn.ModuleList()
        current_channels = self.embed_dim
        target_channels = [512, 256, 128, 64]
        
        for i, out_channels in enumerate(target_channels):
            upsample_block = nn.Sequential(
                nn.ConvTranspose2d(current_channels, out_channels, 
                                 kernel_size=4, stride=2, padding=1),
                nn.BatchNorm2d(out_channels),
                nn.LeakyReLU(0.1, inplace=True),
                nn.Dropout2d(dropout_rate)
            )
            layers.append(upsample_block)
            
            refinement_block = nn.Sequential(
                ResidualBlock(out_channels, use_attention=(i >= 2), dropout_rate=dropout_rate),
                ResidualBlock(out_channels, use_attention=(i >= 2), dropout_rate=dropout_rate)
            )
            layers.append(refinement_block)
            current_channels = out_channels

        final_layers = nn.Sequential(
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.1, inplace=True),
            nn.Dropout2d(dropout_rate),
            ChannelAttention(128),
            nn.Conv2d(128, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.LeakyReLU(0.1, inplace=True),
            nn.Dropout2d(dropout_rate),
            nn.Conv2d(64, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.LeakyReLU(0.1, inplace=True),
            nn.Conv2d(32, self.output_channels, kernel_size=1)
        )
        layers.append(final_layers)
        return nn.Sequential(*layers)
    
    def forward(self, x):
        if x.shape[-2:] != (64, 64):
            raise ValueError(f"Expected input size 64x64, got {x.shape[-2:]}")
        
        if hasattr(self, '_freeze_backbone'):
            with torch.no_grad():
                outputs = self.backbone(x)
        else:
            outputs = self.backbone(x)
            
        features = outputs.last_hidden_state
        spatial_tokens = features[:, self.spatial_start:self.spatial_end, :]
        batch_size = spatial_tokens.shape[0]
        
        spatial_features = spatial_tokens.transpose(1, 2).reshape(
            batch_size, self.embed_dim, self.patches_per_side, self.patches_per_side
        )
        
        depth_map = self.depth_head(spatial_features)
        depth_map = depth_map.squeeze(1)
        depth_map = torch.clamp(depth_map, min=0.0)
        return depth_map
        
    def _initialize_output_layer(self, target_mean=None, target_percentile=None):
        final_conv = None
        for name, module in self.named_modules():
            if isinstance(module, nn.Conv2d):
                final_conv = module
        
        if final_conv is None:
            return
            
        with torch.no_grad():
            nn.init.normal_(final_conv.weight, mean=0.0, std=0.02)
            if final_conv.bias is not None:
                if target_mean is not None:
                    final_conv.bias.fill_(target_mean)
                elif target_percentile is not None:
                    final_conv.bias.fill_(target_percentile)
                else:
                    final_conv.bias.fill_(3.0)

def remap_checkpoint_keys(state_dict, model):
    """
    Handle transformers version skew: some transformers versions nest the
    DINOv3 backbone one level deeper than others
    (backbone.layer.X.* <-> backbone.model.layer.X.*). Remap each checkpoint
    key to whichever variant actually exists on the live model so pretrained
    backbone weights aren't silently dropped.
    """
    model_keys = set(model.state_dict().keys())
    remapped = {}
    for k, v in state_dict.items():
        if k in model_keys:
            remapped[k] = v
        elif k.startswith('backbone.model.') and k.replace('backbone.model.', 'backbone.', 1) in model_keys:
            remapped[k.replace('backbone.model.', 'backbone.', 1)] = v
        elif k.startswith('backbone.') and k.replace('backbone.', 'backbone.model.', 1) in model_keys:
            remapped[k.replace('backbone.', 'backbone.model.', 1)] = v
        else:
            remapped[k] = v
    return remapped

def load_model_for_inference(checkpoint_path, device='cuda', use_multi_gpu=True, data_config=None):
    print(f"Loading model from: {checkpoint_path}")
    checkpoint = torch.load(checkpoint_path, map_location='cpu')
    model_config = checkpoint['model_config']
    
    # Reads from HF_TOKEN env var (e.g. set via `huggingface-cli login` or export HF_TOKEN=...)
    token = os.environ.get("HF_TOKEN")
    model = DINOv3DepthHead(
        model_name=model_config['model_name'],
        freeze_backbone=True,
        token=token,
        input_bands=data_config['input_bands']
    )
    
    remapped_state_dict = remap_checkpoint_keys(checkpoint['model_state_dict'], model)
    missing, unexpected = model.load_state_dict(remapped_state_dict, strict=False)
    if missing or unexpected:
        print(f"⚠️  load_state_dict: {len(missing)} missing key(s), {len(unexpected)} unexpected key(s)")
        if missing:
            print(f"   missing (first 5): {missing[:5]}")
        if unexpected:
            print(f"   unexpected (first 5): {unexpected[:5]}")
    else:
        print("✅ All checkpoint keys matched after remapping")
    model = model.to(device)
    
    if use_multi_gpu and torch.cuda.device_count() > 1:
        num_gpus = torch.cuda.device_count()
        print(f"🚀 Setting up DataParallel across {num_gpus} GPUs!")
        model = nn.DataParallel(model, device_ids=list(range(num_gpus)))
    
    model.eval()
    print(f"✅ Model loaded successfully!")
    return model

# ==========================================
# 4. PROCESSING FUNCTIONS
# ==========================================

def convert_predictions_to_decimeters_int16(predictions):
    nodata_mask = predictions == -9999
    valid_predictions = predictions.copy()
    valid_predictions[nodata_mask] = 0 
    decimeters = valid_predictions * 10
    rounded = np.round(decimeters)
    converted_predictions = rounded.astype(np.int16)
    converted_predictions[converted_predictions < 0] = 0 
    converted_predictions[nodata_mask] = -9999
    return converted_predictions

def save_predictions_as_geotiff(predictions_decimeters, reference_tif_path, output_path):
    ref_ds = gdal.Open(reference_tif_path)
    geotransform = ref_ds.GetGeoTransform()
    projection = ref_ds.GetProjection()
    height, width = predictions_decimeters.shape
    
    output_array = predictions_decimeters.copy()
    if np.issubdtype(predictions_decimeters.dtype, np.floating):
        output_array = np.where(np.isnan(predictions_decimeters), -9999, predictions_decimeters)
    output_array = output_array.astype(np.int16)
    
    driver = gdal.GetDriverByName('GTiff')
    out_ds = driver.Create(
        output_path, width, height, 1, gdal.GDT_Int16,
        options=['COMPRESS=LZW', 'TILED=YES']
    )
    
    out_ds.SetGeoTransform(geotransform)
    out_ds.SetProjection(projection)
    
    band = out_ds.GetRasterBand(1)
    band.WriteArray(output_array)
    band.SetNoDataValue(-9999)
    band.SetDescription("Canopy Height Model - Values in decimeters")
    band.SetMetadataItem("UNITS", "decimeters")
    
    out_ds.SetMetadataItem("PROCESSING", "DINOv3 model prediction")
    out_ds.SetMetadataItem("NODATA_VALUE", "-9999")
    
    band.FlushCache()
    out_ds.FlushCache()
    
    print(f"Successfully saved predictions to: {output_path}")

def check_output_exists(tif_path, output_dir):
    base_name = os.path.basename(tif_path)
    if base_name.endswith('.tif'):
        output_filename = base_name[:-4] + '.chm.tif'
    else:
        output_filename = base_name + '.chm'
    
    output_path = os.path.join(output_dir, output_filename)
    return os.path.exists(output_path), output_path

def filter_unprocessed_files(tif_list, output_dir):
    unprocessed_files = []
    skipped_count = 0
    for tif_path in tif_list:
        exists, expected_output = check_output_exists(tif_path, output_dir)
        if exists:
            skipped_count += 1
        else:
            unprocessed_files.append(tif_path)
    
    print(f"📊 Found {len(unprocessed_files)} files to process, {skipped_count} already completed")
    return unprocessed_files

def complete_dinov3_inference_pipeline_batch(tif_list, model, data_config, inference_config):
    means, stds = data_config['means'], data_config['stds']
    nir_min, nir_max = inference_config['nir_min'], inference_config['nir_max']
    red_min, red_max = inference_config['red_min'], inference_config['red_max']
    green_min, green_max = inference_config['green_min'], inference_config['green_max']
    
    img_size = inference_config['img_size']
    overlap = inference_config['overlap']
    batch_size = inference_config['batch_size']
    visualize_first = inference_config['visualize_first']
    output_dir = inference_config['output_dir']
    
    device = next(model.parameters()).device
    colors = ['#636363','#fc8d59','#fee08b','#ffffbf','#d9ef8b','#91cf60','#1a9850','#005a32']
    forest_ht_cmap, forest_ht_norm, boundaries = create_custom_binned_colormap(colors, vmax=35)
    
    results = []
    output_h, output_w = None, None
    
    for file_idx, tif_path in enumerate(tif_list, 1):
        print(f"\n🔄 Processing TIF {file_idx}/{len(tif_list)}: {os.path.basename(tif_path)}")
        try:
            # Planet 8-band SuperDove order: 1 Coastal Blue, 2 Blue, 3 Green I,
            # 4 Green, 5 Yellow, 6 Red, 7 Red Edge, 8 NIR (1-indexed, rasterio convention)
            with rasterio.open(tif_path) as src:
                nir_band, red_band, green_band = src.read(8), src.read(6), src.read(4)
                nrg_array = np.stack([nir_band, red_band, green_band], axis=0).astype(np.float32)
            
            # Planet SR products use 0 as the nodata/fill value outside the scene footprint
            nrg_array[nrg_array == 0] = np.nan
            
            # Minmax scale to [0,1] using (0, mean+3std) Planet-derived bounds, clip to
            # [0,1], then z-score normalize using the means/stds of the SCALED data --
            # mirrors the original Maxar training preprocessing (minmax -> clip -> zscore).
            nrg_array[0] = (nrg_array[0] - nir_min) / (nir_max - nir_min)
            nrg_array[1] = (nrg_array[1] - red_min) / (red_max - red_min)
            nrg_array[2] = (nrg_array[2] - green_min) / (green_max - green_min)
            
            nrg_array = np.clip(nrg_array, 0, 1)
            for i in range(3):
                nrg_array[i] = (nrg_array[i] - means[i]) / stds[i]
            
            xraster = nrg_array.transpose(1, 2, 0)
            tiler_image = Tiler(data_shape=xraster.shape, tile_shape=(img_size, img_size, 3), 
                                channel_dimension=-1, overlap=overlap, mode='reflect')
            
            if file_idx == 1:
                test_input = torch.randn(1, 3, img_size, img_size).to(device)
                with torch.no_grad():
                    test_output = model(test_input)
                    output_h, output_w = test_output.shape[1], test_output.shape[2]
            
            tiler_mask = Tiler(data_shape=(xraster.shape[0], xraster.shape[1], 1),
                               tile_shape=(output_h, output_w, 1), channel_dimension=-1,
                               overlap=overlap, mode='reflect')
            merger = Merger(tiler=tiler_mask, window='triang')
            
            start_time = time.time()
            tile_count = 0
            
            for batch_id, batch_i in tiler_image(xraster, batch_size=batch_size):
                actual_batch_size = len(batch_i)
                input_batch = batch_i.transpose(0, 3, 1, 2).astype('float32')
                input_batch_tensor = torch.nan_to_num(torch.from_numpy(input_batch).to(device), nan=0.0)
                
                with torch.no_grad():
                    y_batch = model(input_batch_tensor)
                    y_batch_numpy = y_batch.cpu().numpy()
                    del input_batch_tensor, y_batch
                    formatted_output = np.expand_dims(y_batch_numpy, axis=-1) if len(y_batch_numpy.shape) == 3 else y_batch_numpy
                        
                for j in range(actual_batch_size):
                    merger.add(batch_id * batch_size + j, formatted_output[j])
                
                tile_count += actual_batch_size
                if tile_count % (batch_size * 50) == 0:
                    torch.cuda.empty_cache()
            
            predictions_raw = merger.merge(unpad=True)
            predictions_meters = np.squeeze(predictions_raw)
            
            original_nodata = np.isnan(nrg_array[0])
            predictions_meters[original_nodata] = -9999
            
            predictions_decimeters = convert_predictions_to_decimeters_int16(predictions_meters)
            
            # Save GeoTIFF
            base_name = os.path.basename(tif_path)
            string = base_name[-46:-10] if len(base_name) >= 46 else base_name[:-4]
            output_tif = os.path.join(output_dir, f'{string}sr-02m.chm.tif')
            save_predictions_as_geotiff(predictions_decimeters, tif_path, output_tif)
            
            # Optional visualization natively in Jupyter
            if visualize_first and file_idx <= 3:
                plt.figure(figsize=(20, 6))
                
                # Original NRG
                plt.subplot(1, 3, 1)
                nrg_display = nrg_array.transpose(1, 2, 0)
                step = max(1, min(nrg_display.shape[0], nrg_display.shape[1]) // 1000)
                nrg_sub = nrg_display[::step, ::step, :]
                nrg_norm = np.zeros_like(nrg_sub)
                for ch in range(3):
                    valid = ~np.isnan(nrg_sub[:, :, ch])
                    if valid.any():
                        vmin, vmax = np.nanpercentile(nrg_sub[:, :, ch][valid], [2, 98])
                        nrg_norm[:, :, ch] = np.clip((nrg_sub[:, :, ch] - vmin)/(vmax - vmin), 0, 1) if vmax > vmin else 0
                plt.imshow(nrg_norm)
                plt.title(f'NRG Composite\n{base_name}')
                plt.axis('off')
                
                # Predicted CHM
                plt.subplot(1, 3, 2)
                pred_display = predictions_meters.copy()
                pred_display[pred_display == -9999] = np.nan
                pred_sub = pred_display[::step, ::step]
                valid_pred = pred_display[~np.isnan(pred_display)]
                if len(valid_pred) > 0:
                    vmin, vmax = np.nanpercentile(valid_pred, [1, 99])
                    im1 = plt.imshow(pred_sub, cmap=forest_ht_cmap, vmin=vmin, vmax=vmax)
                    plt.colorbar(im1, label='Height (m)')
                else:
                    plt.imshow(pred_sub, cmap='viridis')
                plt.title('Predicted CHM (meters)')
                plt.axis('off')
                
                # Histogram
                plt.subplot(1, 3, 3)
                if len(valid_pred) > 0:
                    plt.hist(valid_pred, bins=50, alpha=0.7, edgecolor='black')
                    plt.xlabel('Predicted Height (m)')
                    plt.title(f'Mean: {valid_pred.mean():.2f}m')
                
                plt.tight_layout()
                plt.show()  # Uses plt.show() directly for Jupyter Notebooks
            
            results.append(predictions_meters)
            
        except Exception as e:
            print(f"❌ Error processing {tif_path}: {str(e)}")
            traceback.print_exc()
            results.append(None)
            
    return results

# ==========================================
# 5. MAIN EXECUTION CELL
# ==========================================

print("🌲 DINOv3 Canopy Height Model Inference (Jupyter Version)")
print("=" * 60)

# Fetch files automatically from directory using glob.
# Restricted to the 8-band Planet SR scenes (the ones the stats in
# PLANET_STATS_FILE were computed from) -- the older 4-band Planet ortho
# products have a different band layout and are not covered by this
# normalization.
tif_list = glob.glob(os.path.join(INPUT_DIR, '*_8b_sr.tif'))
print(f"📋 Found {len(tif_list)} 8-band Planet SR TIF files in {INPUT_DIR}")

# Setup data configuration using the Planet-derived stats computed in the
# data-exploration cells above (1M sampled pixels across the 8-band scenes).
# planet_stats['band_names'] order: Coastal Blue, Blue, Green I, Green,
# Yellow, Red, Red Edge, NIR -- so NIR=idx 7, Red=idx 5, Green=idx 3.
planet_stats = np.load(PLANET_STATS_FILE, allow_pickle=True).item()
NIR_IDX, RED_IDX, GREEN_IDX = 7, 5, 3

DATA_CONFIG = {
    'data_name': 'planet_8b_sr_nrg_scaled',
    'input_bands': 'nrg',
    # Means/stds computed on the SCALED [0,1] data (cell 8), matching how the
    # original Maxar training stats were derived (minmax -> clip -> stats).
    'means': np.load(PLANET_SCALED_MEANS_FILE),
    'stds': np.load(PLANET_SCALED_STDS_FILE),
}

INFERENCE_CONFIG = {
    'device': 'cuda' if torch.cuda.is_available() else 'cpu',
    'batch_size': BATCH_SIZE,
    'img_size': IMG_SIZE,
    'overlap': OVERLAP,
    'visualize_first': VISUALIZE_FIRST,
    'output_dir': OUTPUT_DIR,
    'nir_min': 0.0, 'nir_max': float(planet_stats['mean_plus_3std'][NIR_IDX]),
    'red_min': 0.0, 'red_max': float(planet_stats['mean_plus_3std'][RED_IDX]),
    'green_min': 0.0, 'green_max': float(planet_stats['mean_plus_3std'][GREEN_IDX]),
}

print(f"NRG minmax bounds (0, mean+3std): "
      f"NIR=[0, {INFERENCE_CONFIG['nir_max']:.1f}], "
      f"Red=[0, {INFERENCE_CONFIG['red_max']:.1f}], "
      f"Green=[0, {INFERENCE_CONFIG['green_max']:.1f}]")
print(f"NRG z-score (scaled-space) means={DATA_CONFIG['means']}, stds={DATA_CONFIG['stds']}")

if not tif_list:
    print("❌ No files found in the specified directory!")
else:
    unprocessed_files = filter_unprocessed_files(tif_list, INFERENCE_CONFIG['output_dir'])
    
    if not unprocessed_files:
        print("✅ All files have already been processed!")
    else:
        model = load_model_for_inference(
            CHECKPOINT_PATH, 
            INFERENCE_CONFIG['device'], 
            use_multi_gpu=True,
            data_config=DATA_CONFIG
        )
        
        if model is not None:
            batch_start_time = time.time()
            print(f"\n🚀 Starting batch inference on {len(unprocessed_files)} unprocessed files...")
            
            batch_results = complete_dinov3_inference_pipeline_batch(
                tif_list=unprocessed_files,
                model=model,
                data_config=DATA_CONFIG,
                inference_config=INFERENCE_CONFIG
            )
            
            total_batch_time = time.time() - batch_start_time
            print(f"\n🎉 BATCH PROCESSING COMPLETE!")
            print(f"⏰ Total time: {total_batch_time/60:.1f} minutes")